# Fitts's Law: Predicting How Fast We Can Point

**CS474: Human Computer Interaction — The Way We Interact**

In 1954, Paul Fitts discovered something remarkable: the time to move your hand (or a mouse pointer) to a target is predicted by a simple formula involving only the target's **distance** D and **width** W:

$$ MT = a + b \cdot \log_2\!\left(\frac{D}{W} + 1\right) $$

The log term is the **index of difficulty (ID)**, measured in bits.  Fitts's law is one of the few genuinely quantitative laws in HCI — it's why buttons at screen edges are fast to hit, why context menus beat menu bars, and why tiny close buttons infuriate everyone.

In this notebook you will:

1. Simulate a pointing experiment across a range of target distances and widths
2. Fit Fitts's law with linear regression and interpret a (reaction time) and b (device throughput)
3. Use the fitted model to *predict* and compare real design choices

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(474)

## Part 1: Simulate a Pointing Experiment

A participant clicks targets in a classic Fitts task.  We cross 4 distances with 3 widths (12 conditions, 15 trials each), and generate movement times from the law plus human noise.

In [ ]:
a_true, b_true = 0.20, 0.15   # seconds, seconds/bit — typical mouse values
distances = [64, 128, 256, 512]   # px
widths = [8, 24, 64]              # px
TRIALS = 15

rows = []
for D in distances:
    for W in widths:
        ID = np.log2(D / W + 1)
        mt = a_true + b_true * ID + rng.normal(0, 0.04, TRIALS)
        for m in mt:
            rows.append((D, W, ID, max(m, 0.05)))

data = np.array(rows)  # columns: D, W, ID, MT
print(f"{len(data)} pointing trials, ID range "
      f"{data[:,2].min():.2f}-{data[:,2].max():.2f} bits")

## Part 2: Fit the Law

Fitts's law says MT is *linear in ID*, so a straight-line fit recovers a and b.

In [ ]:
ID, MT = data[:, 2], data[:, 3]
b_hat, a_hat = np.polyfit(ID, MT, 1)
print(f"a = {a_hat*1000:.0f} ms (start-up cost: reaction + click)")
print(f"b = {b_hat*1000:.0f} ms/bit  ->  throughput = {1/b_hat:.1f} bits/s")

plt.figure(figsize=(7, 4))
plt.plot(ID, MT, 'o', ms=4, alpha=0.35, label='trials')
xs = np.linspace(ID.min(), ID.max(), 50)
plt.plot(xs, a_hat + b_hat * xs, '-', color='tab:red', lw=2,
         label=f'fit: MT = {a_hat:.2f} + {b_hat:.2f} x ID')
plt.xlabel('index of difficulty (bits)'); plt.ylabel('movement time (s)')
plt.title("Fitts's law: movement time vs. index of difficulty")
plt.legend()
plt.show()

The slope b characterizes the *pointing device + muscle group*: mice are around 100-200 ms/bit, touchscreens faster for large targets, trackpoints slower.  Comparing 1/b ("throughput") across devices is exactly how input devices are benchmarked (ISO 9241-9).

## Part 3: Use the Model to Critique Designs

Once fitted, the law answers design questions *before* you build anything.

In [ ]:
def predict_mt(D, W):
    return a_hat + b_hat * np.log2(D / W + 1)

designs = [
    ("Small close button, far corner",        800, 12),
    ("Same button, doubled in size",          800, 24),
    ("Button moved near the pointer",         200, 12),
    ("Edge-of-screen target (infinite width)", 800, 200),  # edge lets you overshoot
]
print(f"{'design':42s} {'D(px)':>6} {'W(px)':>6} {'predicted MT':>13}")
for name, D, W in designs:
    print(f"{name:42s} {D:6d} {W:6d} {predict_mt(D, W)*1000:10.0f} ms")

Notice that halving distance and doubling width buy similar savings — and the screen edge (which effectively gives a target enormous width, since the pointer stops there) is the fastest of all.  This is why macOS puts the menu bar at the very top of the screen.

## Your Turn

1. **Run it on yourself.**  Try an interactive Fitts task in your browser (search for "Fitts's law demo" — several university demos exist), or time yourself clicking near/far bookmarks.  Is your personal b larger or smaller than our simulated mouse?
2. **Touchscreen thumbs.**  On phones, targets near the screen center are easy for thumbs but corners are hard — the *effective* D depends on grip.  How would you redesign the experiment in Part 1 to measure a one-handed phone user?
3. **An exception.**  Fitts's law covers *pointing*, not *steering* through constrained paths (that's the steering law) or keyboard shortcuts (no pointing at all).  Name one interaction in an app you use daily where making the target bigger would NOT help, and explain why.
4. **Design connection.**  Pick one affordance/signifier problem your group identified in the activity.  Does Fitts's law suggest the problem is *motor* (target too small/far) or *cognitive* (target not noticeable)?  How would you tell the difference experimentally?

## Reflection

Fitts's original 1954 paper ("The information capacity of the human motor system in controlling the amplitude of movement") framed human movement as an information channel — an idea straight from Shannon.  It remains one of the most-cited results in HCI, and you have now reproduced its core analysis.